# Research MVP: one mechanics-clarification test

Select a **T4 GPU**, run the cells in order, and stop after one full-history seed-0 episode. The only intentional agent change from the supplied v2 baseline is the `v2-mechanics` primer. Qwen revision, FP16, greedy decoding, game settings, budget 80, reread price 2, and generation settings stay fixed. No ingredient locations or seed-specific solution is supplied.

The prior v2 seed-0 run took about **203 seconds of episode time plus 101 seconds of model loading**, excluding setup and idle allocation. This is a cost reference, not a guarantee. No additional episodes or memory comparisons run automatically.

Use the ready-to-run copy with REPO_REF filled in, or enter the supplied commit SHA. The existing v2 baseline stays in its original Drive folder.

In [ ]:
from pathlib import Path
import json
import os
import re
import subprocess
import sys
import uuid

REPO_URL = "https://github.com/sebace05/64610-Memory.git"
REPO_REF = "PASTE_COMMITTED_CODE_SHA"
REPO_DIR = Path("/content/64610-Memory-mechanics-checkpoint")
SETTINGS = {
    "backend": "transformers", "strategy": "full_history", "actor_prompt": "v2-mechanics",
    "model_id": "Qwen/Qwen3-4B-Instruct-2507",
    "model_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
    "dtype": "float16", "recheck_cost": 2, "action_budget": 80,
    "context_limit": 8192, "action_max_new_tokens": 128, "generation_seed": 0,
}
FIRST_SEED = 0  # Matched to the existing v2 baseline; only the mechanics primer changes.


## 1. Check out the selected source revision

An existing checkout is reused only if clean and already at the requested commit. No reset or overwrite is performed.

In [ ]:
if not re.fullmatch(r"[0-9a-f]{40}", REPO_REF):
    raise ValueError("Set REPO_REF to the full pushed implementation commit SHA first.")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", REPO_REF], check=True)
status = subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True)
source_sha = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if status or source_sha != REPO_REF:
    raise RuntimeError("Existing checkout is dirty or at a different revision; preserve it and use a fresh REPO_DIR.")
print("Source commit:", source_sha)

## 2. Persist results before launching inference

Google Drive authorization happens through Colab. Every new launch directory preserves earlier attempts, including partial/error traces. Drive writes may add overhead to episode wall time; model latency is recorded separately.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
PILOT_DIR = Path("/content/drive/MyDrive/64610-Memory/qwen-pilots") / uuid.uuid4().hex
PILOT_DIR.mkdir(parents=True, exist_ok=False)
print("Persistent pilot directory:", PILOT_DIR)

## 3. Install an isolated inference environment and run tests

The separate environment avoids conflicts with Colab's preinstalled Transformers, torchvision, or audio packages. This setup downloads the pinned CUDA-enabled PyTorch distribution; it may take several minutes. No model weights are loaded yet. Test output and the dependency freeze are saved in a fresh checks directory inside PILOT_DIR. The shared Python helper applies the Java/Py4J startup fix.

In [ ]:
import shutil

if shutil.which("java") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "openjdk-17-jre-headless"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.9.11"], check=True)
runtime = Path("/content/memory-qwen-venv")
if not runtime.exists():
    subprocess.run([sys.executable, "-m", "uv", "venv", "--seed", str(runtime)], check=True)
PYTHON = str(runtime / "bin/python")
ENV = dict(os.environ, HF_HOME="/content/hf-cache")
subprocess.run([PYTHON, "-m", "pip", "install", "-r", "requirements-model.txt"], cwd=REPO_DIR, env=ENV, check=True)

def run_module(module, *arguments):
    result = subprocess.run([PYTHON, "-m", module, *map(str, arguments)],
                            cwd=REPO_DIR, env=ENV, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.returncode:
        print(result.stdout)
    result.check_returncode()
    return result.stdout

print(run_module("game_memory.pilot", "--output-dir", PILOT_DIR, "--check"))


## 4. Prepare and validate the pilot

The repository validates GPU support and saves an immutable manifest without loading weights. Settings live in SETTINGS above; the model revision and FP16 precision match the original T4 run. Rerunning this cell accepts identical settings and rejects changes without overwriting the original manifest. Use a fresh pilot directory for a different configuration.

In [ ]:
def run_diagnostic(seeds, *, prepare_only=False):
    arguments = []
    for key, value in SETTINGS.items():
        arguments.extend(["--" + key.replace("_", "-"), str(value)])
    if prepare_only:
        arguments.append("--prepare-only")
    return run_module("game_memory.pilot", "--output-dir", PILOT_DIR,
                      *arguments, "--seeds", *seeds)

print(run_diagnostic([FIRST_SEED], prepare_only=True))


## 5. Run one real episode

Weights load once per invocation. A valid game loss returns exit code zero; model/context/infrastructure errors return nonzero and preserve their traces. Do not add constrained decoding or hidden history to improve this checkpoint without documenting the change.

SETTINGS["actor_prompt"] explicitly selects the policy version. v2-mechanics adds the inventory/preparation primer to the existing v2 planning and repair instructions; v1/v2 remain available for the preserved baselines. Do not mix versions within a pilot directory. An improvement in actual Qwen completion remains to be measured.


In [ ]:
print(run_diagnostic([FIRST_SEED]))


## 6. Audit the checkpoint and inspect the trace

The report independently checks budgets, tokens, request/response pairing, labels, and completion. It keeps infrastructure errors separate from gameplay losses. A technically valid report does not demonstrate adequate gameplay capability. Open the saved events.jsonl to inspect exact prompts and decisions.

In [ ]:
output = run_module("game_memory.summarize_diagnostics", PILOT_DIR)
(PILOT_DIR / "checkpoint-report.json").write_text(output)
report = json.loads(output)
for result in report["episodes"]:
    print(json.dumps({key: result[key] for key in
        ("episode_id", "actor_prompt", "seed", "status", "success", "termination_reason", "steps",
         "recipe_rechecks", "budget_spent", "input_tokens", "output_tokens", "runtime_seconds")}, indent=2))
    print("Maximum score:", result["behavior"]["maximum_observed_score"])
print("Logging checks passed:", report["logging_checks_passed"])
print("Infrastructure checks passed:", report["infrastructure_checks_passed"])


## Save the checkpoint for review

Download one bundle containing the manifest, dependency versions, checks, launch logs, result, and exact trace. Return that bundle for inspection before running memory comparisons. A completed execution is not evidence that the model can play successfully. Do not rerun the same seed in this directory; every attempt needs its own preserved folder.

In [ ]:
results = [json.loads(path.read_text()) for path in PILOT_DIR.glob("*/result.json")]
if len(results) != 1:
    raise RuntimeError("This checkpoint expects exactly one episode. Inspect the saved files.")
trace_path = PILOT_DIR / results[0]["episode_id"] / "events.jsonl"
print("Saved trace:", trace_path)


The archive contains research logs only; cached model weights are outside PILOT_DIR.

In [ ]:
from google.colab import files
archive = shutil.make_archive(str(PILOT_DIR.parent / (PILOT_DIR.name + "-checkpoint")), "zip", PILOT_DIR)
files.download(archive)


## Stop here: review the real-model checkpoint

Inspect actual progress, inventory handling, invalid requests, and repeated observations. The earlier seed-0 v2 run still failed despite fewer recipe rereads. If this run also fails, retain it as diagnostic evidence and choose one small, justified next step. Do not start a large tuning sweep or manufacture a memory comparison.

The Friday MVP, after a useful-gameplay checkpoint, is prose vs structured JSON at two reread prices and two development seeds: eight episodes, one compact results table, and two annotated trace windows. Hybrid memory, the 120-episode evaluation, dashboards, and general batch/resume infrastructure are deferred. No remaining experiments run automatically in this notebook.

Disconnect the GPU when finished. Files already written to Drive are preserved. This notebook's orchestration has been tested locally; its new Colab execution still requires you.